# Visão geral do Dataset

O contexto do projeto e o dicionário completo de variáveis estão no `README.md`.

In [ ]:
# Carregando a base 
import pandas as pd

pd.set_option("display.max_columns", 50)

df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')


In [2]:
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")
display(df.head())

Linhas: 7,043 | Colunas: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Tipos de dados

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## Verificando a qualidade dos dados

Observando nulos, duplicatas, tipos, valores inválidos e consistência entre colunas.

In [4]:
nulos = df.isna().sum()
print("Colunas com nulos:", nulos[nulos > 0].to_dict() or "nenhuma")
print("Linhas duplicadas:", df.duplicated().sum())
print("customerID duplicado:", df["customerID"].duplicated().sum())

Colunas com nulos: nenhuma
Linhas duplicadas: 0
customerID duplicado: 0


Não temos valores faltantes e nem duplicatas nos dados.

### O `TotalCharges` deveria ser numérica
O `isna()` não achou nada, mas a coluna é texto. Convertemos com `errors="coerce"` (o que não for número vira `NaN`) para descobrir quais linhas são problemáticas.

In [5]:
tc_num = pd.to_numeric(df["TotalCharges"], errors="coerce")
invalidos = df[tc_num.isna()]

print(f"TotalCharges não numérico: {len(invalidos)} linhas")
display(invalidos[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])
print("Valores de tenure nessas linhas:", invalidos["tenure"].unique())

TotalCharges não numérico: 11 linhas


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


Valores de tenure nessas linhas: [0]


As 11 linhas têm `tenure = 0`. São clientes recém-chegados que ainda não receberam nenhuma cobrança, e nenhum deles deu churn. Não é dado faltante o valor deveria ser  0.

## Verificando as categorias de cada variável

In [9]:
cat_cols = df.select_dtypes(include="str").columns.drop(["customerID", "TotalCharges"])
for c in cat_cols:
    print(f"{c}: {df[c].unique().tolist()}")

gender: ['Female', 'Male']
Partner: ['Yes', 'No']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No phone service', 'No', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'Yes', 'No internet service']
OnlineBackup: ['Yes', 'No', 'No internet service']
DeviceProtection: ['No', 'Yes', 'No internet service']
TechSupport: ['No', 'Yes', 'No internet service']
StreamingTV: ['No', 'Yes', 'No internet service']
StreamingMovies: ['No', 'Yes', 'No internet service']
Contract: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling: ['Yes', 'No']
PaymentMethod: ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Churn: ['No', 'Yes']


## Verificando a consistência entre as colunas

In [ ]:
cols_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                 "TechSupport", "StreamingTV", "StreamingMovies"]

# 'No internet service' deve coincidir exatamente com InternetService == 'No'
sem_net_nas_colunas = (df[cols_internet] == "No internet service").all(axis=1).sum()
print("Clientes com 'No internet service' em todas as 6 colunas:", sem_net_nas_colunas)
print("Clientes com InternetService == 'No':", (df["InternetService"] == "No").sum())

# 'No phone service' deve coincidir com PhoneService == 'No'
print("MultipleLines == 'No phone service':", (df["MultipleLines"] == "No phone service").sum())
print("PhoneService == 'No':", (df["PhoneService"] == "No").sum())

# TotalCharges deveria ser ~ tenure * MonthlyCharges
df_tenure_positivo = df.assign(tc=tc_num).query("tenure > 0")
razao = df_tenure_positivo["tc"] / (df_tenure_positivo["tenure"] * df_tenure_positivo["MonthlyCharges"])
print(f"\nTotalCharges / (tenure*MonthlyCharges): mediana={razao.median():.3f}, "
      f"min={razao.min():.2f}, max={razao.max():.2f}")

Clientes com 'No internet service' em todas as 6 colunas: 1526
Clientes com InternetService == 'No': 1526
MultipleLines == 'No phone service': 682
PhoneService == 'No': 682

TotalCharges / (tenure*MonthlyCharges): mediana=1.000, min=0.69, max=1.57


## Observando o Target

In [11]:
df["Churn"].value_counts(normalize=True).round(3)

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64

### Problemas encontrados 

-  `TotalCharges` é texto e tem espaços em branco (clientes com `tenure == 0`).
-  Nomes de colunas fora do padrão (`customerID`, `StreamingTV`...).
-  "No internet service" / "No phone service" são redundantes com "No".
- `TotalCharges` é quase `tenure × MonthlyCharges`, portanto é muito correlacionada com as outras duas
-  Colunas Yes/No precisam virar 1/0.
-  Target desbalanceado (~73% / 27%): usar estratificação no split e olhar recall/AUC além da acurácia.